In [39]:
# Importing function to load datasets

import sys
sys.path.append("../src")

from data_loader_rachel import load_data

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [41]:
df_demo, df_web, df_experiment = load_data("../data/raw/")

Datasets loaded successfully:
Demo: (70609, 9)
Web: (755405, 5)
Experiment: (70609, 2)


In [42]:
# Double checking

df_demo.head()
df_web.head()
df_experiment.head()

df_demo.shape, df_web.shape, df_experiment.shape

((70609, 9), (755405, 5), (70609, 2))

### Quick reminder

df_demo → Who the client is

df_experiment → Which group they are in (control/test)

df_web → What they actually did

This means:
- Multiple rows per client
- Each row = a user interaction / step / event
- This is our behavioral dataset (very valuable for analysis)

# Analyzing contents of datasets and potential improvements

In [43]:
# Checking out columns and common keys

df_demo.columns
df_experiment.columns
df_web.columns

Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time'], dtype='str')

In [44]:
# Looking at the first few rows

df_demo.head()
df_experiment.head()
df_web.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


In [45]:
# Understanding the structure: client_id is the common denominator in all datasets! Non-null values anywhere!

df_demo.info()
df_experiment.info()
df_web.info()

<class 'pandas.DataFrame'>
RangeIndex: 70609 entries, 0 to 70608
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   client_id         70609 non-null  int64  
 1   clnt_tenure_yr    70595 non-null  float64
 2   clnt_tenure_mnth  70595 non-null  float64
 3   clnt_age          70594 non-null  float64
 4   gendr             70595 non-null  str    
 5   num_accts         70595 non-null  float64
 6   bal               70595 non-null  float64
 7   calls_6_mnth      70595 non-null  float64
 8   logons_6_mnth     70595 non-null  float64
dtypes: float64(7), int64(1), str(1)
memory usage: 4.8 MB
<class 'pandas.DataFrame'>
RangeIndex: 70609 entries, 0 to 70608
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   client_id  70609 non-null  int64
 1   variation  50500 non-null  str  
dtypes: int64(1), str(1)
memory usage: 1.1 MB
<class 'pandas.DataFrame'>
RangeI

### Fixes: date_time was set as str 

This is critical because later we'll need:
- session ordering
- time differences
- funnel analysis

In [46]:
df_web['date_time'] = pd.to_datetime(df_web['date_time'])
df_web.info()

<class 'pandas.DataFrame'>
RangeIndex: 755405 entries, 0 to 755404
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   client_id     755405 non-null  int64         
 1   visitor_id    755405 non-null  str           
 2   visit_id      755405 non-null  str           
 3   process_step  755405 non-null  str           
 4   date_time     755405 non-null  datetime64[us]
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 28.8 MB


### Fixes: Missing values in df_demo

In [47]:
df_demo = df_demo.dropna()
df_demo

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
0,836976,6.0,73.0,60.5,U,2.0,45105.30,6.0,9.0
1,2304905,7.0,94.0,58.0,U,2.0,110860.30,6.0,9.0
2,1439522,5.0,64.0,32.0,U,2.0,52467.79,6.0,9.0
3,1562045,16.0,198.0,49.0,M,2.0,67454.65,3.0,6.0
4,5126305,12.0,145.0,33.0,F,2.0,103671.75,0.0,3.0
...,...,...,...,...,...,...,...,...,...
70604,7993686,4.0,56.0,38.5,U,3.0,1411062.68,5.0,5.0
70605,8981690,12.0,148.0,31.0,M,2.0,101867.07,6.0,6.0
70606,333913,16.0,198.0,61.5,F,2.0,40745.00,3.0,3.0
70607,1573142,21.0,255.0,68.0,M,3.0,475114.69,4.0,4.0


### Fixes: Missing variation in df_experiment

70609 (rows) - 50500 (non-null values in column) = 20109 missing values (where variation is missing)

- Variation tells us whether a client is in a control group or in a test group!
- So if variation is missing, that client was NOT part of the experiment.

In [48]:
df_experiment = df_experiment.dropna(subset=['variation'])
df_experiment

,client_id,variation
0,9988021,Test
1,8320017,Test
2,4033851,Control
3,1982004,Test
4,9294070,Control
...,...,...
50495,393005,Control
50496,2908510,Control
50497,7230446,Test
50498,5230357,Test


### To be aware: There are 14 missing values in df_demo - pretty clean already